# Phase 5 — evidence freeze, final write-up and release

Companion: `PHASE5_PLAN.md`, written 23 October 2026. **Planned, not executed.**
Use a **fresh CPU runtime**. This notebook verifies and packages existing work;
it does not train, re-mine, relabel, retune thresholds, or publish automatically.

C4/C5/C6: **AI-reviewed; not independently human-validated** under
`P4-AI-AUDIT-2026-10-23`. Historical C2/C3 labels were also agent-reviewed;
C1 ≈0–2% is an estimate, not a new 50-pair audit. New audit rates are
12%, 0%, 22%; unsure-inclusive rates are 16%, 0%, 24%. Wilson intervals
do not measure reviewer accuracy.

**Before running:** synchronize the current source changes deliberately. A fresh
clone may not contain the amended scorer/helpers yet; section 1 stops if they
are missing. Never reset a dirty checkout. Keep credentials in Colab Secrets
(`HF_TOKEN`, optional for public reads); never paste them into a cell.


In [ ]:
# P5-0: configuration (no network or external writes)
from pathlib import Path
import os, sys, json, hashlib, subprocess, shutil, csv, re, time, zipfile
from datetime import datetime, timezone

IN_COLAB = Path('/content').is_dir()
SOURCE_ROOT = Path('/content/EmbedEd') if IN_COLAB else Path.cwd()
WORK = (Path('/content/embeded-phase5') if IN_COLAB else
        SOURCE_ROOT / 'artifacts/phase4_exports/2026-10-23/phase5-work')
SOURCE_BRANCH = 'arena/01a0e714-embeded'
SOURCE_REVISION = None  # optional exact Git commit; never used to reset an existing checkout
HF_REPO = 'Kusshal/Embed'
INPUT_REVISION = 'e3144bea7732c20ab2803782282282be4fede7b4'
REPAIR_REVISION = 'a77f8d2d1f26b5168a88822a7d1138f027eee4af'
AMENDMENT = 'P4-AI-AUDIT-2026-10-23'
REPORT = WORK / 'report/measurements.md'
EXPORTS = WORK / 'exports'
def sha(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def need(*paths):
    missing = [str(p) for p in paths if not Path(p).is_file()]
    if missing:
        raise SystemExit('Missing required inputs:\n' + '\n'.join(missing))
def save_new_or_identical(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and path.read_bytes() != data:
        raise SystemExit(f'Preserving changed file: {path}. Use a fresh workspace or reconcile manually.')
    if not path.exists():
        path.write_bytes(data)
print('CPU closure only; source:', SOURCE_ROOT, '; isolated workspace:', WORK)


## 1. Source readiness and environment

Only a missing checkout is cloned. Existing files are preserved. This cell does
not fetch/reset/pull an existing checkout. If it stops on missing amended source,
synchronize the current project changes to this branch or upload the current
source into `/content/EmbedEd`, then retry in a fresh runtime.

Installation uses the repository requirements, with **Colab's existing torch**
rather than a new pinned torch build. No model/checkpoint is downloaded.


In [ ]:
# P5-1: preserve existing source; fail closed on an old checkout
if not SOURCE_ROOT.exists():
    if not IN_COLAB:
        raise SystemExit('Set SOURCE_ROOT to the existing project checkout.')
    subprocess.run(['git', 'clone', '--branch', SOURCE_BRANCH, '--single-branch',
                    'https://github.com/ksu-gitreaper807/EmbedEd', str(SOURCE_ROOT)], check=True)
    if SOURCE_REVISION:
        subprocess.run(['git', '-C', str(SOURCE_ROOT), 'checkout', '--detach', SOURCE_REVISION], check=True)
need(SOURCE_ROOT / 'scripts/phase4_closure.py',
     SOURCE_ROOT / 'scripts/audit_sample.py',
     SOURCE_ROOT / 'notebooks/Phase4_Colab.ipynb',
     SOURCE_ROOT / 'report/phase4_ai_audit_protocol.json',
     SOURCE_ROOT / 'report/phase4_ai_audit_freeze.json')
if 'def review_provenance(' not in (SOURCE_ROOT / 'scripts/audit_sample.py').read_text():
    raise SystemExit('Old scorer: synchronize the amended AI-review source before continuing.')
if 'AI-reviewed; not independently human-validated' not in (SOURCE_ROOT / 'notebooks/Phase4_Colab.ipynb').read_text():
    raise SystemExit('Old table builder: synchronize the updated Phase 4 notebook.')
if (SOURCE_ROOT / '.git').exists():
    actual_commit = subprocess.check_output(['git', '-C', str(SOURCE_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
    dirty = subprocess.check_output(['git', '-C', str(SOURCE_ROOT), 'status', '--porcelain'], text=True)
else:
    actual_commit, dirty = None, 'source tree synchronized without Git metadata'
if SOURCE_REVISION and actual_commit != SOURCE_REVISION:
    raise SystemExit('Existing source cannot certify requested Git revision; not resetting it.')
os.chdir(SOURCE_ROOT)
sys.path.insert(0, str(SOURCE_ROOT))
WORK.mkdir(parents=True, exist_ok=True)
EXPORTS.mkdir(parents=True, exist_ok=True)
source_files = sorted({p for directory in ('embeded', 'scripts', 'demo')
                       for p in (SOURCE_ROOT / directory).rglob('*.py')
                       if '__pycache__' not in p.parts})
source_files += [SOURCE_ROOT / p for p in (
    'scripts/run_all.sh', 'embeded/requirements.txt', 'notebooks/Phase4_Colab.ipynb',
    'notebooks/Phase5_Colab.ipynb', 'PHASE5_PLAN.md') if (SOURCE_ROOT / p).is_file()]
source_files += sorted((SOURCE_ROOT / 'report/gate_g1').glob('*.json'))
SOURCE_MANIFEST = {'commit': actual_commit, 'dirty': bool(dirty),
                   'files': {str(p.relative_to(SOURCE_ROOT)): sha(p) for p in source_files}}
save_new_or_identical(WORK / 'source_manifest.json',
                      (json.dumps(SOURCE_MANIFEST, indent=2) + '\n').encode())
print('Source commit:', actual_commit, '; dirty:', bool(dirty))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                str(SOURCE_ROOT / 'embeded/requirements.txt')], check=True)


## 2. Pinned, selective evidence download

Public reads only, pinned to the verified audit commit (which retains repaired
benchmark files). The download includes **all 37 benchmark prediction files**,
not just seeds 13–15: the integrity gate checks the whole published inventory.
No `.pt` checkpoint or embedding matrix is requested.

The snapshot remains immutable. Copies go into the separate Phase 5 workspace;
the source checkout's report is never overwritten. If the Hub returns 429,
honour Retry-After and stop retrying for the recorded cooldown.


In [ ]:
# P5-2: exact allowlist; no broad pull and no writer action
from huggingface_hub import snapshot_download

TOKEN = os.environ.get('HF_TOKEN')
if IN_COLAB and not TOKEN:
    try:
        from google.colab import userdata
        TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        TOKEN = None
bench_names = ['C0_13'] + [f'C{c}_{seed}' for c in range(1, 7) for seed in range(13, 19)]
sprime_names = [f'sprime_C{c}_{seed}' for c in range(1, 7) for seed in range(13, 16)]
sprime_names += ['sprime_transfer_C0_13']
required = [f'artifacts/runs/{name}/{file}' for name in bench_names
            for file in ('eval_metrics.json', 'predictions.npz')]
required += [f'artifacts/runs/{name}/eval_metrics.json' for name in sprime_names]
required += [f'artifacts/audit_c{c}/{file}' for c in (4, 5, 6)
             for file in ('audit_key.csv', 'audit_pairs.md', 'audit_labels.csv', 'audit_result.json')]
required += [f'artifacts/triples_C{c}.jsonl' for c in (4, 5, 6)]
required += ['artifacts/fragments.jsonl', 'artifacts/pairs_test.tsv', 'artifacts/phase4_decisions.json']
audit_reports = ['phase4_ai_audit_protocol.json', 'phase4_ai_audit_freeze.json',
                 'phase4_ai_audit_results.json', 'phase4_audit_completion.md',
                 'phase4_ai_review_c4.csv', 'phase4_ai_review_c5.csv', 'phase4_ai_review_c6.csv',
                 'phase4_audit_outcome_log.md']
required += ['report/' + name for name in audit_reports] + ['report/measurements.md']
optional = [f'artifacts/runs/{name}/run_config.json' for name in bench_names + sprime_names]
optional += ['report/figures/umap_panels.png', 'report/figures/umap_params.json']
SNAPSHOT = Path(snapshot_download(repo_id=HF_REPO, repo_type='dataset',
                revision=INPUT_REVISION, allow_patterns=required + optional,
                token=TOKEN or False, max_workers=2))
need(*(SNAPSHOT / name for name in required))
for name in required + optional:
    src = SNAPSHOT / name
    if not src.is_file():
        continue
    if name == 'report/measurements.md':
        dest = WORK / 'input_measurements.md'
    elif name == 'report/phase4_audit_outcome_log.md':
        dest = WORK / 'sanitycheck.md'
    else:
        dest = WORK / name
    save_new_or_identical(dest, src.read_bytes())
if not REPORT.exists():
    REPORT.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(WORK / 'input_measurements.md', REPORT)
INPUT_MANIFEST = {'repo': HF_REPO, 'revision': INPUT_REVISION,
                  'files': {name: sha(SNAPSHOT / name) for name in required + optional
                            if (SNAPSHOT / name).is_file()}}
save_new_or_identical(WORK / 'input_manifest.json',
                      (json.dumps(INPUT_MANIFEST, indent=2) + '\n').encode())
def verify_input_copies():
    for name, digest in INPUT_MANIFEST['files'].items():
        if name == 'report/measurements.md':
            dest = WORK / 'input_measurements.md'
        elif name == 'report/phase4_audit_outcome_log.md':
            dest = WORK / 'sanitycheck.md'
        else:
            dest = WORK / name
        if sha(dest) != digest or sha(SNAPSHOT / name) != digest:
            raise SystemExit('Pinned working input changed: ' + name)
verify_input_copies()
os.environ['EMBEDED_ARTIFACTS'] = str(WORK / 'artifacts')
os.environ['EMBEDED_REPORT'] = str(REPORT)
os.environ['EMBEDED_AUDIT_PROTOCOL'] = str(WORK / 'report/phase4_ai_audit_protocol.json')
os.environ['EMBEDED_AUDIT_FREEZE'] = str(WORK / 'report/phase4_ai_audit_freeze.json')
from embeded import settings as S
S.ARTIFACTS, S.REPORT_MD = WORK / 'artifacts', REPORT
print('Pinned inputs verified present:', len(INPUT_MANIFEST['files']))


## 3. Fresh acceptance checks

This executes the actual offline suite, not a hard-coded historical test count.
It then checks all benchmark replays, frozen audit provenance/outcomes, and the
300 code fragments behind the reviewed pairs. A failure is a stop, not an
instruction to re-train, re-mine or relax the gate.


In [ ]:
# P5-3: tests, syntax and evidence gates
verify_input_copies()
result = subprocess.run([sys.executable, '-m', 'pytest', '-q', 'embeded/tests'],
                        cwd=SOURCE_ROOT, env={**os.environ, 'HF_HUB_OFFLINE': '1'},
                        text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
(WORK / 'offline_tests.log').write_text(result.stdout)
print(result.stdout[-5000:])
if result.returncode:
    raise SystemExit('Offline suite failed. See offline_tests.log; do not release.')
subprocess.run(['bash', '-n', str(SOURCE_ROOT / 'scripts/run_all.sh')], check=True)
from scripts.phase4_closure import audit_mismatches, verified_audits
from scripts.audit_sample import sample_pairs
mismatches = audit_mismatches()
if mismatches:
    raise SystemExit(f'Benchmark replay discrepancies: {mismatches}')
AUDITS = verified_audits(WORK / 'sanitycheck.md')
summary = json.loads((WORK / 'report/phase4_ai_audit_results.json').read_text())
fragments = {int(row['idx']): row['text'] for row in
             map(json.loads, (S.ARTIFACTS / 'fragments.jsonl').read_text().splitlines())}
for cond in ('C4', 'C5', 'C6'):
    d = S.ARTIFACTS / f'audit_{cond.lower()}'
    key = list(csv.DictReader((d / 'audit_key.csv').open()))
    expected = sample_pairs(50, 13, conds=(cond,))
    actual = [dict(id=r['id'], condition=r['condition'], anchor=int(r['anchor']),
                   negative=int(r['negative'])) for r in key]
    if actual != expected or summary[cond] != AUDITS[cond]:
        raise SystemExit(f'{cond}: sample identity or summary disagreement')
    sections = re.split(r'^## ([a-z0-9-]+)\n', (d / 'audit_pairs.md').read_text(), flags=re.M)
    for i, row in enumerate(key, 1):
        blocks = re.findall(r'```java\n(.*?)\n```', sections[2*i], re.S)
        if blocks != [fragments[int(row['anchor'])].strip(), fragments[int(row['negative'])].strip()]:
            raise SystemExit(f'Code/key mismatch: {row["id"]}')
ACCEPTANCE = {'passed_utc': datetime.now(timezone.utc).isoformat(),
              'benchmark_runs': len(bench_names), 'mismatches': 0,
              'verified_audits': list(AUDITS), 'reviewed_fragments_verified': 300,
              'test_log_sha256': sha(WORK / 'offline_tests.log'),
              'source_manifest_sha256': sha(WORK / 'source_manifest.json'),
              'input_manifest_sha256': sha(WORK / 'input_manifest.json')}
(WORK / 'acceptance.json').write_text(json.dumps(ACCEPTANCE, indent=2) + '\n')
print(json.dumps(ACCEPTANCE, indent=2))


## 4. Generate the existing three-panel table

Reuse the tested Phase 4 builder, rather than maintaining another implementation.
Its original decisions are read as historical inputs—not endorsement of stale
causal framing. C1 stays the headline; the method/claim note below supersedes
older interpretation. Exact 13–15 seeds supply the primary table; all 37
benchmark runs were checked above.

Do not confuse candidate rejection percentages (C5 0.27%, C6 13.6%) with
audited residual contamination. Both old C2/C3 and new C4/C5/C6 are agent/AI
review, not human confirmation.


In [ ]:
# P5-4: call the tested Phase 4 table seam after all current gates
verify_input_copies()
need(WORK / 'acceptance.json')
if audit_mismatches():
    raise SystemExit('Replay integrity changed since acceptance.')
AUDITS = verified_audits(WORK / 'sanitycheck.md')
DECISIONS = json.loads((S.ARTIFACTS / 'phase4_decisions.json').read_text())['decisions']
ROOT = str(WORK)  # the reused builder reads the downloaded score-bound outcome log
def replace_or_append(heading, block):
    old = REPORT.read_text()
    pattern = re.compile(re.escape(heading) + r'.*?(?=\n## |\Z)', re.S)
    new = pattern.sub(lambda _: block.rstrip() + '\n', old) if pattern.search(old) else old.rstrip() + '\n\n' + block
    REPORT.write_text(new.rstrip() + '\n')
phase4 = json.loads((SOURCE_ROOT / 'notebooks/Phase4_Colab.ipynb').read_text())
builders = [''.join(c['source']) for c in phase4['cells']
            if c['cell_type'] == 'code' and '# --- 5b.' in ''.join(c['source'])]
if len(builders) != 1:
    raise SystemExit('Expected exactly one tested Phase 4 table-builder cell.')
exec(compile(builders[0], 'Phase4_Colab.ipynb#5b', 'exec'), globals())
method_note = """## Phase 5 — interpretation and reviewer provenance

All C4/C5/C6 rates are AI-reviewed; not independently human-validated
(P4-AI-AUDIT-2026-10-23). Historical C2/C3 labels were also agent-reviewed,
with different review-pass/granularity conventions; C1 is a historical estimate.
These are descriptive comparisons, not a reviewer-controlled causal test.
Wilson intervals cover sampling uncertainty only, not AI accuracy or bias.

C4: 6/50, 12% [5.6%, 23.8%], unsure-inclusive 16%.
C5: 0/50, 0% [0%, 7.1%], unsure-inclusive 0%.
C6: 11/50, 22% [12.8%, 35.2%], unsure-inclusive 24%.
C6's point estimate is below its 25–35% prediction band but its interval overlaps
that band. Preserve scorer-state-first endpoint ordering; neither mechanism
explanation is independently established. Retain C1 as headline.

The 0.27% and 13.6% supply figures count candidates rejected by the lexical cut,
not contamination remaining. The Jaccard comparison is with the anchor.
C3's unseen mean can exceed the untuned C0 mean; no significance or equality
claim follows merely from nearby means. The published ~0.95 gap is not fully
explained by identifying protocol/model differences.
"""
replace_or_append('## Phase 5 — interpretation and reviewer provenance', method_note)
# Generate the manuscript's table dependencies from the same pinned inputs.
subprocess.run([sys.executable, '-m', 'scripts.phase5_report', '--snapshot', str(WORK),
                '--output', str(WORK / 'report/final')], cwd=SOURCE_ROOT, check=True)
subprocess.run([sys.executable, '-m', 'scripts.export_confusion_matrices', '--snapshot', str(WORK),
                '--out-dir', str(WORK / 'report/figures/confusion_matrices')], cwd=SOURCE_ROOT, check=True)
canonical = (WORK / 'report/final/tables.md').read_text()
canonical = canonical.replace('## Phase 5 — verified primary results', '## Phase 4 — the one table')
canonical = canonical.replace('Primary benchmark:', '### Panel A — benchmark\n\nPrimary benchmark:')
canonical = canonical.replace('### Generalisation', '### Panel B — generalisation')
canonical = canonical.replace('### Audit evidence', '### Panel C — audit evidence')
replace_or_append('## Phase 4 — the one table', canonical)
current_note = ('Current generated tables supersede older narrative summaries. All-negative F1=0; '
                'all-positive F1=.2406424, not a ceiling. Near-one thresholds do not imply '
                'constant-negative predictions. Primary C2 predicts 48.1–87.3% positive; '
                'C3 33.6–42.7%. Historical audit labels are agent review, not human validation. '
                's′ seen F1 uses the threshold-selection validation slice, not an independent test.\n\n')
if not REPORT.read_text().startswith('> Phase 5 current-results notice:'):
    REPORT.write_text('> Phase 5 current-results notice: ' + current_note + REPORT.read_text())
labels = [int(line.rstrip().split('\t')[-1]) for line in
          (S.ARTIFACTS / 'pairs_test.tsv').open() if line.strip()]
if not labels or not set(labels) <= {0, 1}:
    raise SystemExit('Invalid test labels for prevalence/baseline verification.')
prevalence = sum(labels) / len(labels)
BASELINE = {'n': len(labels), 'positive': sum(labels), 'prevalence': prevalence,
            'all_negative_f1': 0.0, 'all_positive_f1': 2*prevalence/(1+prevalence)}
(WORK / 'baseline_check.json').write_text(json.dumps(BASELINE, indent=2) + '\n')
(WORK / 'phase5_decisions.json').write_text(json.dumps({
    'headline': 'C1', 'primary_seeds': [13, 14, 15], 'layout': 'three panels',
    'mechanism': 'dose/similarity not isolated by amended AI audit',
    'new_experiments': False, 'publication_approved': False,
    'input_revision': INPUT_REVISION, 'table_sha256': sha(REPORT)
}, indent=2) + '\n')
print('Generated report:', REPORT, '; SHA256:', sha(REPORT))
print('Measured baselines:', BASELINE)


## 5. Editorial work — deliberate stop

The next cell creates **draft copies**, a claim ledger and a false-by-default
review checklist. It does not silently rewrite scientific conclusions.
Edit these drafts in place, reconcile every task in PHASE5_PLAN §5, and mark
a claim reviewed only after checking its evidence.

Required corrections include the all-positive baseline, C3 MAP@R, exact seed
denominators, voided C3=C4 equality, historical AI-review attribution, the
candidate-rejection/contamination distinction, the C3-versus-C0 unseen means,
and the unproved explanation of the published 0.95 gap.


In [ ]:
# P5-5: create editable drafts without overwriting earlier editorial work
DRAFT = WORK / 'draft'
DRAFT.mkdir(exist_ok=True)
for source, name in [('PROJECT_REPORT.md', 'PROJECT_REPORT.md'), ('paper/paper.tex', 'paper.tex'),
                     ('paper/IEEEtran.cls', 'IEEEtran.cls'),
                     ('TIMELINE.md', 'TIMELINE.md'), ('HANDOFF.md', 'HANDOFF.md'),
                     ('FUTURE_PROSPECTS.md', 'FUTURE_PROSPECTS.md'),
                     ('report/phase4_audit_completion.md', 'audit_completion.md')]:
    need(SOURCE_ROOT / source)
    target = DRAFT / name
    if not target.exists():
        shutil.copy2(SOURCE_ROOT / source, target)
ledger_path = WORK / 'claim_ledger.json'
if not ledger_path.exists():
    claims = [
        ('benchmark', 'Report exact generated F1/MAP@R and seed denominators.', ['report/measurements.md']),
        ('audit', 'All reviewed audit rates are agent/AI review, not independent human validation.',
         ['report/phase4_ai_audit_results.json', 'report/phase4_audit_completion.md']),
        ('mechanism', 'Audit evidence does not isolate false-negative dose from similarity.',
         ['sanitycheck.md', 'report/phase4_ai_audit_results.json']),
        ('generalisation', 'Compare actual unseen means with C0, without inventing significance or ties.',
         ['report/measurements.md']),
        ('baseline', 'All-negative F1 is zero; all-positive F1 is 2*pi/(1+pi).', ['baseline_check.json']),
        ('published_gap', 'Protocol/model differences are a limitation, not proof resolving the full gap.',
         ['draft/paper.tex'])
    ]
    ledger_path.write_text(json.dumps([dict(id=i, statement=s, evidence=e, reviewed=False)
                                       for i, s, e in claims], indent=2) + '\n')
REVIEW_FIELDS = ['numbers_reconciled', 'audit_attribution_correct', 'claims_reconciled',
                 'report_reviewed', 'references_checked', 'figures_checked', 'paper_compiled',
                 'paper_visually_reviewed', 'no_secrets', 'release_target_reviewed']
review_path = WORK / 'phase5_review.json'
if not review_path.exists():
    review_path.write_text(json.dumps({'reviewer': '', 'checks': {k: False for k in REVIEW_FIELDS},
                                      'reviewed_sha256': {}}, indent=2) + '\n')
print('EDIT these before release:', DRAFT, ledger_path, review_path)
print('These files are drafts, not a completed review or publication.')


## 6. Compile and visually inspect the paper

If `pdflatex` is already available, the cell runs it twice and saves the actual
log. It does not install a TeX distribution. Otherwise compile the draft using
an available LaTeX editor/compiler, save `draft/paper.pdf` and the real compiler
log at `latex_compile.log`, then review the PDF visually.

A successful compiler does not establish scientific accuracy or correct layout.
Check that required figures actually appear; guarded missing-image includes can
otherwise let an incomplete manuscript compile.


In [ ]:
# P5-6: compile only when a compiler is available; no automatic TeX installation
compiler = shutil.which('pdflatex')
if not compiler:
    raise SystemExit('No pdflatex installed. Compile the saved draft with an available LaTeX compiler; preserve PDF and real log before release.')
logs = []
for _ in range(2):
    proc = subprocess.run([compiler, '-interaction=nonstopmode', '-halt-on-error', 'paper.tex'],
                          cwd=DRAFT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    logs.append(proc.stdout)
    (WORK / 'latex_compile.log').write_text('\n'.join(logs))
    if proc.returncode:
        raise SystemExit('LaTeX failed. Repair the draft; see latex_compile.log.')
need(DRAFT / 'paper.pdf')
print('Compiled PDF:', DRAFT / 'paper.pdf', '; visual review is still required.')


## 7. Review and release-package gate

Run the next cell to see the exact files requiring sign-off. After actual review,
set the checklist fields and claim-ledger entries to true, identify the reviewer,
and copy the displayed SHA-256 mapping into `reviewed_sha256`.
Changing a reviewed document invalidates its sign-off.

This is a content-hash-bound review record, not an automated truth or visual
quality checker. No release package is certified while required review is missing.


In [ ]:
# P5-7: enumerate the exact editorial deliverables, then verify sign-off
RELEASE_DOCS = [
    'report/measurements.md', 'draft/PROJECT_REPORT.md', 'draft/paper.tex', 'draft/paper.pdf',
    'draft/TIMELINE.md', 'draft/HANDOFF.md', 'claim_ledger.json',
    'report/phase4_ai_audit_protocol.json', 'report/phase4_ai_audit_freeze.json',
    'report/phase4_ai_audit_results.json', 'report/phase4_audit_completion.md',
    'report/phase4_ai_review_c4.csv', 'report/phase4_ai_review_c5.csv',
    'report/phase4_ai_review_c6.csv', 'source_manifest.json', 'input_manifest.json',
    'acceptance.json', 'offline_tests.log', 'latex_compile.log', 'baseline_check.json',
    'phase5_decisions.json', 'sanitycheck.md']
RELEASE_DOCS += ['report/final/' + name for name in (
    'results.json', 'tables.md', 'numbers.tex', 'benchmark.tex',
    'generalisation.tex', 'audits.tex', 'extension.tex')]
RELEASE_DOCS += ['draft/FUTURE_PROSPECTS.md', 'draft/audit_completion.md']
RELEASE_DOCS += ['draft/IEEEtran.cls']
RELEASE_DOCS += [str(p.relative_to(WORK)) for p in
                 sorted((WORK / 'report/figures/confusion_matrices').rglob('*')) if p.is_file()]
need(*(WORK / name for name in RELEASE_DOCS))
CURRENT_REVIEW_HASHES = {name: sha(WORK / name) for name in RELEASE_DOCS}
print('After actual review, record these hashes in phase5_review.json:')
print(json.dumps(CURRENT_REVIEW_HASHES, indent=2))
review = json.loads((WORK / 'phase5_review.json').read_text())
if not review.get('reviewer', '').strip():
    raise SystemExit('Reviewer identity and actual review are required.')
if any(review.get('checks', {}).get(k) is not True for k in REVIEW_FIELDS):
    raise SystemExit('Required review is incomplete; checklist defaults to false.')
if review.get('reviewed_sha256') != CURRENT_REVIEW_HASHES:
    raise SystemExit('Review hashes missing/stale. Re-review changed deliverables.')
claims = json.loads((WORK / 'claim_ledger.json').read_text())
if not claims or any(c.get('reviewed') is not True or not c.get('statement') or not c.get('evidence') for c in claims):
    raise SystemExit('Unreviewed/unsupported claim-ledger entries remain.')
for claim in claims:
    need(*(WORK / name for name in claim['evidence']))
if not (DRAFT / 'paper.pdf').read_bytes().startswith(b'%PDF-'):
    raise SystemExit('Compiled PDF is missing or invalid.')
print('Editorial sign-off complete for these exact file contents.')


## 8. Final recheck and local evidence package

This creates a local ZIP only. It rechecks review hashes, source/input hashes,
the offline suite, replay/audit gates and shell syntax first. It includes only
explicit deliverables, exact source files and audit evidence—not caches,
credentials, checkpoints or the whole workspace.

**No Git or HF upload is performed.** Review the diff/targets, obtain owner
publication approval, then publish from the consolidating writer with a current
parent guard. Record and download-verify the resulting Git/HF commits separately.
Preserve the package before releasing this notebook's own CPU runtime.


In [ ]:
# P5-8: fail closed even when Colab Run-all continued after an earlier error
review = json.loads((WORK / 'phase5_review.json').read_text())
if (not review.get('reviewer', '').strip()
        or any(review.get('checks', {}).get(k) is not True for k in REVIEW_FIELDS)):
    raise SystemExit('Release blocked: incomplete editorial review.')
need(*(WORK / name for name in RELEASE_DOCS))
current = {name: sha(WORK / name) for name in RELEASE_DOCS}
if current != review.get('reviewed_sha256'):
    raise SystemExit('Release blocked: reviewed content has changed.')
claims = json.loads((WORK / 'claim_ledger.json').read_text())
if not claims or any(c.get('reviewed') is not True for c in claims):
    raise SystemExit('Release blocked: claim ledger is incomplete.')
verify_input_copies()
for name, expected in SOURCE_MANIFEST['files'].items():
    if sha(SOURCE_ROOT / name) != expected:
        raise SystemExit('Source changed after acceptance; restart verification in a new workspace.')
for name, expected in INPUT_MANIFEST['files'].items():
    if sha(SNAPSHOT / name) != expected:
        raise SystemExit('Pinned snapshot changed unexpectedly.')
if audit_mismatches():
    raise SystemExit('Benchmark integrity changed.')
verified_audits(WORK / 'sanitycheck.md')
again = subprocess.run([sys.executable, '-m', 'pytest', '-q', 'embeded/tests'],
                       cwd=SOURCE_ROOT, env={**os.environ, 'HF_HUB_OFFLINE': '1'},
                       text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
(WORK / 'release_tests.log').write_text(again.stdout)
if again.returncode:
    raise SystemExit('Release tests failed; see release_tests.log.')
subprocess.run(['bash', '-n', str(SOURCE_ROOT / 'scripts/run_all.sh')], check=True)
if not (DRAFT / 'paper.pdf').read_bytes().startswith(b'%PDF-'):
    raise SystemExit('Release blocked: invalid PDF.')
payload = {name: WORK / name for name in RELEASE_DOCS + ['phase5_review.json', 'release_tests.log']}
for c in (4, 5, 6):
    for name in ('audit_key.csv', 'audit_pairs.md', 'audit_labels.csv', 'audit_result.json'):
        rel = f'artifacts/audit_c{c}/{name}'
        payload[rel] = WORK / rel
for name in SOURCE_MANIFEST['files']:
    payload['source/' + name] = SOURCE_ROOT / name
manifest = {name: sha(p) for name, p in payload.items()}
package = EXPORTS / ('phase5_release_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ') + '.zip')
with zipfile.ZipFile(package, 'x', zipfile.ZIP_DEFLATED) as bundle:
    for name, path in payload.items():
        bundle.write(path, name)
    bundle.writestr('release_manifest.json', json.dumps(manifest, indent=2) + '\n')
with zipfile.ZipFile(package) as bundle:
    if bundle.testzip() is not None:
        raise SystemExit('Archive CRC failure.')
    for name, digest in manifest.items():
        if hashlib.sha256(bundle.read(name)).hexdigest() != digest:
            raise SystemExit('Archive hash mismatch: ' + name)
print('LOCAL PACKAGE VERIFIED:', package)
print('SHA256:', sha(package))
print('Not published. Owner approval and immutable Git/HF release verification remain separate.')


## Stop condition and optional later research

Phase 5 ends when the verified table and reconciled documents are reviewed,
the PDF is compiled/visually checked, and an owner-approved release has been
published and hash-verified. A local package alone is **not** a published release.

Human replication, denoising δ-ladders, FN-weighted objectives and a second
encoder are possible later proposals. Each needs its own explicit scope,
algorithm/rubric, budget and pre-recorded endpoints. Reaching this cell does
not authorize any of them. Do not keep a GPU allocated for writing.
